In [ ]:
%%capture
!pip install unsloth
# Get the latest nightly Unsloth!
!pip uninstall unsloth -y && pip install --upgrade --no-cache-dir --no-deps git+https://github.com/unslothai/unsloth.git

# **Loading the Model**

In [ ]:
from unsloth import FastLanguageModel
import torch
max_seq_length = 2048 # Choose any! it auto supports RoPE Scaling internally
dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = True # Use 4bit quantization to reduce memory usage. Can be False.

fourbit_models = [
    "unsloth/Meta-Llama-3.1-8B-bnb-4bit",
    "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit",
    "unsloth/Meta-Llama-3.1-70B-bnb-4bit",
    "unsloth/Meta-Llama-3.1-405B-bnb-4bit",
    "unsloth/Mistral-Nemo-Base-2407-bnb-4bit",
    "unsloth/Mistral-Nemo-Instruct-2407-bnb-4bit",
    "unsloth/mistral-7b-v0.3-bnb-4bit",
    "unsloth/mistral-7b-instruct-v0.3-bnb-4bit",
    "unsloth/Phi-3.5-mini-instruct",
    "unsloth/Phi-3-medium-4k-instruct",
    "unsloth/gemma-2-9b-bnb-4bit",
    "unsloth/gemma-2-27b-bnb-4bit",
] # More models at https://huggingface.co/unsloth

model, tokenizer = FastLanguageModel.from_pretrained(
    # model_name = "unsloth/mistral-7b-v0.3-bnb-4bit",
      model_name = "unsloth/Meta-Llama-3.1-8B-bnb-4bit",
    # model_name = "mrbmaryam/LSH_ParsLite", # Mistral Fine-tuned
    # model_name = "mrbmaryam/LLAMA_LSH_ParsLite", # LLAMA Fine-tuned
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

**Add LoRA adapters to update 1 to 10% of all parameters**

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,  # support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

# **Data Preparation**

In [ ]:
alpaca_prompt = """You are a log parsing assistant. Your task is to identify variable elements within the logs, generalize these elements, and construct a template that represents the structure of these log messages. Please, follow the instruction and return an accurate response.


### Instruction:
Extract the template for the following log message. Replace any variable element with the placeholder '<*>'. Do not provide any explanation, only return the template.

### Input:
{}

### Response:
{}"""


EOS_TOKEN = tokenizer.eos_token # Must add EOS_TOKEN

def formatting_prompts_func(examples):
    inputs       = examples["RepresentativeLog"]
    outputs      = examples["EventTemplate"]
    texts = []

    for  input, output in zip( inputs, outputs):

        text = alpaca_prompt.format( input, output) + EOS_TOKEN
        texts.append(text)

    return { "text" : texts, }
pass

from datasets import load_dataset

dataset = load_dataset("mrbmaryam/Train_one_per_temp_lsh", split = "train")
dataset = dataset.map(formatting_prompts_func, batched = True,)

# **Training with Energy Consumption**




<a name="Train"></a>
### Train the model
Training is done using Huggingface TRL's `SFTTrainer`! More docs here: [TRL SFT docs](https://huggingface.co/docs/trl/sft_trainer). It is done in 60 steps to speed things up, but you can set `num_train_epochs=1` for a full run, and turn off `max_steps=None`. We also support TRL's `DPOTrainer`!

In [ ]:
#@title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

Install PyJoules for monitoring the energy consumption

In [ ]:
!pip install pyJoules

In [ ]:
import time
import pandas as pd
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

# PyJoules imports
from pyJoules.energy_meter import measure_energy
from pyJoules.device.nvidia_device import NvidiaGPUDomain
from pyJoules.handler.csv_handler import CSVHandler

# =====================================================================
# Set up CSV logging
# =====================================================================
csv_handler = CSVHandler('fine-tuning_energy_log_5.csv')

# =====================================================================
# Define the training function wrapped by PyJoules
# =====================================================================
@measure_energy(domains=[NvidiaGPUDomain(0)], handler=csv_handler)
def run_training():
    """Run fine-tuning with GPU energy monitoring via PyJoules."""
    trainer = SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        train_dataset=dataset,
        dataset_text_field="text",
        max_seq_length=max_seq_length,
        dataset_num_proc=2,
        packing=False,
        args=TrainingArguments(
            per_device_train_batch_size=2,
            gradient_accumulation_steps=8,
            warmup_steps=5,
            max_steps=60,
            learning_rate=2e-4,
            fp16=not is_bfloat16_supported(),
            bf16=is_bfloat16_supported(),
            logging_steps=1,
            optim="adamw_8bit",
            weight_decay=0.01,
            lr_scheduler_type="linear",
            seed=3407,
            output_dir="outputs",
            report_to="none",
        ),
    )
    trainer_stats = trainer.train()
    return trainer_stats

# =====================================================================
# Run training and measure total time
# =====================================================================
print("Starting training with PyJoules GPU power logging...")
start_time = time.time()

trainer_stats = run_training()

end_time = time.time()
training_duration = end_time - start_time
print(f"Training finished in {training_duration:.2f} seconds")

# =====================================================================
# Save and analyze energy data
# =====================================================================
csv_handler.save_data()
print("Energy data saved to 'fine-tuning_energy_log.csv'.")

In [ ]:
import pandas as pd

# Read CSV with correct delimiter
df = pd.read_csv('fine-tuning_energy_log_5.csv', sep=';')

# Find GPU column automatically
gpu_col = [c for c in df.columns if 'nvidia' in c.lower() or 'gpu' in c.lower()]
if not gpu_col:
    raise ValueError("No NVIDIA GPU energy column found in fine-tuning_energy_log.csv.")
gpu_col = gpu_col[0]

# Convert to numeric just in case
df[gpu_col] = pd.to_numeric(df[gpu_col], errors='coerce')

# 1. Get raw energy in Milli-Joules (mJ)
total_energy_mJ = df[gpu_col].sum()

# 2. Convert to Joules (J) for standard reporting
total_energy_joules = total_energy_mJ / 1000

# 3. Calculate Power in Watts (Joules / Seconds)
average_power_watts = total_energy_joules / training_duration

# 4. Calculate Power in Milli-watts (optional, for comparison)
average_power_mW = total_energy_mJ / training_duration

print(f"\n=== Fine-tuning GPU Energy Report ===")
print(f"GPU column used: {gpu_col}")
print(f"Total Energy: {total_energy_mJ:.4f} mJ")      # Corrected Label
print(f"Total Energy: {total_energy_joules:.4f} J")    # Standard Unit
print(f"Average Power: {average_power_watts:.2f} W")   # Corrected Math
print(f"Training Duration: {training_duration:.2f} s")

# **Save Model on HuggingFace**

In [ ]:
import getpass
import os

os.environ["HUGGING_FACE_HUB_TOKEN"] = getpass.getpass("Token: ")
assert os.environ["HUGGING_FACE_HUB_TOKEN"]

Token: ··········


In [ ]:
from transformers import AutoModel, AutoTokenizer

# Assuming you have already trained your model and tokenizer
model_name = "----"
token = "----"

# Push model to Hugging Face Hub
model.push_to_hub(model_name)

# Push tokenizer to Hugging Face Hub
tokenizer.push_to_hub(model_name)

print(f"Model and tokenizer successfully pushed to {model_name} on Hugging Face Hub.")
